<a href="https://colab.research.google.com/github/jesobczyk/NustecSchool2026/blob/main/L3_HartreeFock/L3_SkyrmeHF.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Lecture 3 — your own Hartree–Fock solver

In Lecture 2 the nucleus was a **Woods–Saxon well**: a guessed potential with a depth, a radius and a surface, fitted to (e,e′p) data. Here the well is **not guessed**. It is made by the nucleons themselves, from a nuclear force, by the Hartree–Fock loop of Lecture 2:

$$\rho(\mathbf r)=\sum_{\alpha\,\in\,\text{occ}}|\varphi_\alpha(\mathbf r)|^2
\;\longrightarrow\; U[\rho](\mathbf r)
\;\longrightarrow\; \Big[-\nabla\cdot\frac{\hbar^2}{2m^*(\mathbf r)}\nabla+U(\mathbf r)+\text{spin–orbit}\Big]\varphi_\alpha=\varepsilon_\alpha\varphi_\alpha
\;\longrightarrow\;\text{new }\rho \;\longrightarrow\;\dots$$

repeated until nothing changes. The solver in `hf_core.py` does exactly this, for spherical nuclei, in about a second.

**What you will see**

- **A**: the loop converging, and what happens when you remove the piece of the force that gives saturation.
- **B**: the density and the mean field it makes, and the effective mass of Lecture 1.
- **C**: the orbitals, the magic numbers with and without spin–orbit, and a comparison with (e,e′p).
- **D**: the orbitals in momentum space, against a local Fermi gas with the same density, and the spectral-function stripes of Lecture 2.
- **E**: a nucleon leaving through the same field, against a plane wave.

Run the cell below once, then each panel. If the sliders do not appear, set `L.USE_WIDGETS = False` and every panel draws static figures instead.

In [ ]:
#@title ▶ Run this cell first (sets up the notebook on Google Colab; does nothing on your own computer)
import os, sys
if 'google.colab' in sys.modules:
    if not os.path.isdir('/content/NustecSchool2026'):
        !git clone -q https://github.com/jesobczyk/NustecSchool2026.git /content/NustecSchool2026
    %cd /content/NustecSchool2026/L3_HartreeFock
    from google.colab import output
    output.enable_custom_widget_manager()
%matplotlib inline

In [ ]:
import hf_l3 as L       # panels;  the physics is in hf_core.py
import hf_core as H
# L.USE_WIDGETS = False  # uncomment if the sliders do not appear

## The force: Skyrme in five terms

A realistic nuclear force (Lecture 3, chiral EFT) is too expensive for a one-second calculation. The **Skyrme force** is a cheap effective force with the same physics in a few terms. Each one has a job:

| term | what it is | what it does |
|---|---|---|
| $t_0$ | zero-range attraction | binds the nucleus |
| $t_3\,\rho^{\alpha}$ | repulsion that grows with density | **saturation**: stops the collapse (it stands in for the three-nucleon force) |
| $t_1,\,t_2$ | momentum dependence | the **effective mass** $m^*/m \approx 0.7$ of Lecture 1, and the surface |
| $W_0$ | spin–orbit | splits $j=\ell\pm\tfrac12$: the **magic numbers** of Lecture 2 |
| Coulomb | protons only | pushes the proton levels up |

Because the force has zero range, the energy is an integral of a function of three local densities — the density $\rho$, the kinetic density $\tau$ and the spin–orbit density $J$:

$$E=\int d^3r\;\mathcal H\big(\rho(\mathbf r),\,\tau(\mathbf r),\,J(\mathbf r)\big),$$

and the mean field is its derivative, $U_q=\delta E/\delta\rho_q$. Two parameter sets are included: **SLy4** (Chabanat et al. 1998) and **SkM\*** (Bartel et al. 1982). Both were fitted to binding energies and radii of a few doubly magic nuclei and to nuclear matter — not to the nucleus you are computing.

## A — The loop

The calculation starts from a Woods–Saxon guess, builds the density, the fields, new orbitals, and repeats. Each iteration mixes 40 % of the new density into the old one, which keeps the loop stable.

Left: the total energy, with the measured binding energy. Middle: the size of the nucleus. Right: the energies of the occupied orbitals.

Then move the $t_3$ slider down: you remove part of the repulsion that grows with density.

**Questions**

1. How many iterations does ¹⁶O need? Which quantity settles first?
2. Compare binding energies and charge radii with experiment for ¹²C, ¹⁶O, ⁴⁰Ca and ²⁰⁸Pb. Which is worst, and why might that be?
3. Set $t_3 \times 0.9$, then $0.75$, then $0$. What happens to the energy and the radius? What does this say about saturation — the constant density $\rho_0$ of Lecture 2?

In [ ]:
L.show_A()

## B — The density and the field it makes

Left: neutron, proton and total densities. The interior density is close to $\rho_0 = 0.16$ fm⁻³, the value every nucleus shares (Lecture 2).

Middle: the mean field $U(r)$ for neutrons and protons, against the Woods–Saxon guess the loop started from. Nobody told the solver the depth or the radius.

Right: the effective mass $m^*(r)/m$, from the momentum-dependent terms $t_1, t_2$. Inside the nucleus a nucleon moves as if it were lighter: this is the momentum-dependent potential of Lecture 1 (GiBUU) seen from another side.

**Questions**

4. What is the central depth of the field for ⁴⁰Ca? How does it compare with the 50–60 MeV of the Woods–Saxon wells of Lecture 2?
5. Why is the proton field less deep than the neutron field, and why does it not go to zero outside?
6. What is $m^*/m$ in the interior for SLy4 and for SkM\*? Where does it go back to 1, and why?

In [ ]:
L.show_B()

## C — Orbitals and magic numbers

Each spherical orbital is labelled $(n,\ell,j)$ and holds $2j+1$ nucleons (Lecture 2). Thick lines are occupied, dashed lines are empty. For ¹²C, the protons are compared with the (e,e′p) removal energies of Mougey et al. (1976).

**Questions**

7. ⁴⁰Ca: set $W_0 \times 0$. Which levels become degenerate? Which shell gaps survive, and which disappear?
8. ¹²C: compare the proton 1p3/2 and 1s1/2 with (e,e′p). Which is closer? Is the 1s problem of Lecture 2 (it needs a deeper well) still there with a self-consistent field?
9. In ¹²C only the 1p3/2 is filled, not its partner 1p1/2. Why is ¹²C still well described by a spherical calculation, while e.g. ²⁰Ne would not be?

In [ ]:
L.show_C()

## D — Momentum space and the spectral function

The momentum distribution of each orbital is the Fourier (Bessel) transform of its radial function (Lecture 2):

$$\tilde\varphi_{n\ell j}(p)=\sqrt{\tfrac{2}{\pi}}\int_0^\infty dr\;r\,u_{n\ell j}(r)\,j_\ell(pr/\hbar c),\qquad
n(p)=\frac{1}{4\pi}\sum_{\text{occ}}(2j+1)\,|\tilde\varphi_{n\ell j}(p)|^2 .$$

Middle: the total against a **local Fermi gas built from the same density**, $n(k)=\frac{1}{4\pi^3}\int d^3r\,\theta\big(k_F(r)-k\big)$ with $k_F(r)=(3\pi^2\rho_q(r))^{1/3}$. Right: the spectral function of the orbitals — each orbital is a stripe at its removal energy $-\varepsilon_\alpha$, smeared by a width $\Gamma$.

**Questions**

10. Where does the HF momentum distribution have strength that the local Fermi gas does not? Is there anything above $\approx 300$ MeV in either?
11. Neither has the strength that L2's missing third must carry. Why can no Hartree–Fock calculation, however good the force, produce it?

In [ ]:
L.show_D()

## E — A nucleon leaving through the same field

In (e,e′p) and in neutrino scattering the struck nucleon has to leave the nucleus. The simplest consistent picture: it moves through **the same mean field** that bound it (this is the starting point of the Ghent continuum calculations). The panel solves the radial equation at positive energy $T$ in the HF field and compares it with a plane wave, both normalised to the same amplitude far away. The difference far away is a phase shift; inside the nucleus the wave is also distorted.

**Questions**

12. For ⁴⁰Ca and ℓ = 0, compare $T$ = 50, 150 and 300 MeV. When is the plane wave a good approximation inside the nucleus?
13. This field is real: nothing is absorbed. What is missing compared with the optical potential and the transparency of Lecture 2?

In [ ]:
L.show_E()

## Under the hood

Everything above comes from one function. Try your own nucleus (`(Z, N)`), your own parameters, or print the energy decomposition:

In [ ]:
res = H.run_hf('40Ca', 'SLy4')
print(res['status'], res['iterations'], 'iterations')
for k, v in res['energy'].items(): print(f'{k:>15s}: {v:9.1f} MeV')
print([(o['label'], round(o['e'], 1)) for o in res['levels']['n'] if o['occ'] > 0])

**How it is solved.** On a radial grid ($h$ = 0.05 fm, up to 12–16 fm) the equation for $u(r)=rR(r)$,
$$-\big(B\,u'\big)'+\Big[B\,\frac{\ell(\ell+1)}{r^2}+\frac{B'}{r}+U_q+\frac{W_q}{r}\big(j(j+1)-\ell(\ell+1)-\tfrac34\big)\Big]u=\varepsilon\,u,\qquad B=\frac{\hbar^2}{2m^*(r)},$$
becomes a symmetric tridiagonal matrix, whose lowest eigenvalues are the orbital energies. Levels are filled from the bottom; densities are mixed between iterations. Limits of this solver: spherical symmetry, no pairing (open-shell nuclei are approximate), no time-odd terms, centre-of-mass correction only at the one-body level. With SLy4 it reproduces the published ¹⁶O, ⁴⁰Ca and ²⁰⁸Pb binding energies within about 1 MeV.